In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error
from datetime import date
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import logging
import math

import matplotlib.pyplot as plt
from datetime import datetime
from matplotlib.ticker import PercentFormatter
plt.style.use('fivethirtyeight')


In [ ]:
final_prediction_2023_with_test_data = pd.read_excel('.././outputs/final_prediction_2023_with_test_data.xlsx')
final_prediction_2023 = pd.read_excel('.././outputs/almacenes_si_predictions_2023.xlsx')
rmse_por_llave_resultados_df = pd.read_excel('.././outputs/rmse_por_llave_resultados.xlsx')
data_by_week = pd.read_parquet('./datasets/2_curated/almacenes_si_curated_by_week.parquet')


In [ ]:
ventas_por_llave = final_prediction_2023_with_test_data.groupby(['llave'], as_index=False)['y_true'].sum()
rmse_con_venta_real = rmse_por_llave_resultados_df.merge(ventas_por_llave, how = 'left', on = 'llave')
rmse_con_venta_real.head()

In [ ]:
rmse_con_venta_real['venta_minus_rmse'] = rmse_con_venta_real['y_true'] - rmse_con_venta_real['rmse']
rmse_con_venta_real['venta_plus_rmse'] = rmse_con_venta_real['y_true'] + rmse_con_venta_real['rmse']
rmse_con_venta_real['venta_minus_rmse'] = rmse_con_venta_real['venta_minus_rmse'].astype(int)
rmse_con_venta_real['venta_plus_rmse'] = rmse_con_venta_real['venta_plus_rmse'].astype(int)

In [ ]:
# q1,q2,q3,q4 = rmse_con_venta_real['rmse'].quantile([0.25,0.5,0.75,1])
# print(f'Quantile 1: {q1}')
# print(f'Quantile 2: {q2}')
# print(f'Quantile 3: {q3}')
# print(f'Quantile 4: {q4 }')

In [ ]:
family_codes_keep = ['240','239','244','245','247','246','242','202','243','219','230','277','214','252','250','251','281','248','268','209','280','264','265','266','201','267']

In [ ]:
rmse_con_venta_real['family_code'] = rmse_con_venta_real['llave'].apply(lambda x: x[:3])
rmse_con_venta_real = rmse_con_venta_real[rmse_con_venta_real['family_code'].isin(family_codes_keep)]
rmse_con_venta_real[rmse_con_venta_real['rmse'] >= rmse_con_venta_real['rmse'].mean()]

In [ ]:
rmse_con_venta_real['proporcion_error_porcentual'] = round((rmse_con_venta_real['rmse'] / rmse_con_venta_real['y_true'])*100,2)

In [ ]:
rmse_con_venta_real[rmse_con_venta_real['rmse'] == rmse_con_venta_real['rmse'] .max()]

In [ ]:
rmse_con_venta_real.to_excel('.././outputs/proporcion_error_porcentual.xlsx', index = False)

In [175]:
data_with_porcentual_change_more_50 = rmse_con_venta_real[(rmse_con_venta_real['proporcion_error_porcentual'] > 50)]
# data_with_porcentual_change_more_50 = rmse_con_venta_real[(rmse_con_venta_real['proporcion_error_porcentual'] > 50) & (rmse_con_venta_real['rmse'] > rmse_con_venta_real['rmse'].mean())]
# data_with_porcentual_change_more_50 = rmse_con_venta_real[(rmse_con_venta_real['cambio_porcentual'] < 50) & (rmse_con_venta_real['rmse'] > rmse_con_venta_real['rmse'].mean())]
data_with_porcentual_change_more_50.sort_values(by = ['proporcion_error_porcentual'], ascending=False, inplace=True)
data_with_porcentual_change_more_50

,llave,rmse,y_true,venta_minus_rmse,venta_plus_rmse,family_code,proporcion_error_porcentual
138,219339CB2,60.716244,1,-59,61,219,6071.62
420,240BQ4PC9,62.558975,2,-60,64,240,3127.95
347,240AW1OL3,30.744878,1,-29,31,240,3074.49
255,239BV6NL5,16.311053,1,-15,17,239,1631.11
1410,250310462CJ8,12.993505,1,-11,13,250,1299.35
...,...,...,...,...,...,...,...
222,239BB8OS7,3.087022,6,2,9,239,51.45
533,245BT4,17.992071,35,17,52,245,51.41
205,239AL1OC9,5.620103,11,5,16,239,51.09
510,243BB8MZ6,30.771525,61,30,91,243,50.45


In [ ]:
data_with_porcentual_change_more_50.head(5).to_clipboard(index = False)

In [ ]:
data_by_week_train = data_by_week[data_by_week['date_week'] < '2023-01-01']

In [ ]:
final_prediction_2023_with_test_data.head(1)

In [ ]:
def plot_train_test_sales_comparisson(final_prediction_2023_with_test_data , data_by_week_train, llave, save_fig = False):
    plt.style.use('fivethirtyeight')
    llave_to_explore = llave
    train_data = data_by_week_train[data_by_week_train['combination'] == llave_to_explore]
    test_data = final_prediction_2023_with_test_data[final_prediction_2023_with_test_data['llave'] == llave_to_explore]
    test_data['ds'] = pd.to_datetime(test_data['ds'])
    forescast_key = train_data['combination'].iat[0]


    x_train_value = train_data['date_week'].values
    y_train_value = train_data['quantity'].values
    x_test_value = test_data['ds'].values
    y_test_value = test_data['y_true'].values

    # Setting size of our plot
    fig, ax = plt.subplots(figsize=(20,6))
        
    # Plotting each occupation category
    plt.plot(x_train_value, y_train_value , color = '#000000', lw=2, alpha = 0.7, marker=".", ms=10)
    plt.plot(x_test_value , y_test_value, color = '#FF004D', lw=2, alpha = 0.7, marker=".", ms=10)

    # X and y labels
    plt.ylabel('quantity', fontsize=10, color='#414141',labelpad=15)
    plt.xlabel('Date', fontsize=10, color='#414141',labelpad=15)

    # Bolded horizontal line at y=0
    plt.axhline(y=0, color='#414141', linewidth=1.5, alpha=.5)

    plt.axhline(y=np.mean(y_train_value), color='#39A7FF', linewidth=1.5, alpha=.9,linestyle="--")
    plt.text(x=pd.to_datetime('2022-06-01'), y=max(y_train_value), s = "AVG(Ventas) Train", fontsize=12.5, fontweight='bold', color='#39A7FF');

        
    # Y-labels to only these
    plt.yticks( fontsize=10, color='#414141');
    plt.xticks( fontsize=10, color='#414141');
    plt.title(f"Ventas Train VS Test\nLlave: {forescast_key}\n",color='#414141', loc='left', fontweight='bold', );
    if save_fig:
        plt.savefig(f'./figures/high_rmse/{forescast_key}.png', bbox_inches = 'tight', dpi = 100);



In [ ]:
for i in tqdm(data_with_porcentual_change_more_50['llave'].values):
    plot_train_test_sales_comparisson(final_prediction_2023_with_test_data , data_by_week_train, llave = i, save_fig=True)

In [ ]:
top_10_keys_porcentual_changes = data_with_porcentual_change_more_50['llave'].head(10) # get the top 10 extreme porcentual change

for i in top_10_keys_porcentual_changes:
    plot_train_test_sales_comparisson(final_prediction_2023_with_test_data , data_by_week_train, llave = i, save_fig=True)

In [ ]:
for i in top_10_keys_porcentual_changes:
    dummy = final_prediction_2023_with_test_data[final_prediction_2023_with_test_data['llave'] == i]
    display(dummy[['ds','y_true','yhat']])